In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
consumer_df = pd.read_csv("../data/raw/consumer_dataset_raw.csv")
population_df = pd.read_csv("../data/raw/us_ev_population_raw.csv")
iea_df = pd.read_csv("../data/raw/iea_global_ev_raw.csv")
stock_df = pd.read_csv("../data/raw/stock_details_raw.csv")
sales_df = pd.read_excel("../data/raw/global_ev_growth.xlsx")
charging_df = pd.read_csv("../data/raw/openchargemap_global_raw.csv")
survey_df = pd.read_csv("../data/raw/survey_details.csv")

/var/folders/fp/w6yn5n6d4cb299k1w5hy5xgc0000gn/T/ipykernel_91314/3521166484.py:6: DtypeWarning: Columns (36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  charging_df = pd.read_csv("../data/raw/openchargemap_global_raw.csv")


### IEA EV Dataset Cleaning:

In [15]:
iea_df.columns

Index(['region', 'category', 'parameter', 'mode', 'powertrain', 'year', 'unit',
       'value'],
      dtype='object')

In [4]:
iea_df.columns = iea_df.columns.str.lower()

iea_clean = iea_df[
    iea_df["parameter"].isin([
        "EV sales",
        "EV stock",
        "EV charging points"
    ])
]

In [5]:
iea_clean = iea_clean[
    iea_clean["mode"] == "Cars"
]

iea_clean = iea_clean[[
    "region",
    "parameter",
    "powertrain",
    "year",
    "value"
]]

In [6]:
iea_clean.isna().sum()

region        0
parameter     0
powertrain    0
year          0
value         0
dtype: int64

In [20]:
print("Original rows:", iea_df.shape[0])
print("Filtered rows:", iea_clean.shape[0])

Original rows: 12654
Filtered rows: 2975


In [7]:
iea_clean.to_csv("../data/processed/iea_ev_clean.csv", index=False)

### EV Population Dataset Cleaning:

In [9]:
population_df.columns

Index(['vin (1-10)', 'county', 'city', 'state', 'postal code', 'model year',
       'make', 'model', 'electric vehicle type',
       'clean alternative fuel vehicle (cafv) eligibility', 'electric range',
       'legislative district', 'dol vehicle id', 'vehicle location',
       'electric utility', '2020 census tract'],
      dtype='object')

In [11]:
population_df.columns = population_df.columns.str.lower()

ev_population_clean = population_df.groupby(
    ["model year", "make", "model", "electric vehicle type"]
).size().reset_index(name="vehicle_count")

In [12]:
ev_population_clean.isna().sum()

model year               0
make                     0
model                    0
electric vehicle type    0
vehicle_count            0
dtype: int64

In [13]:
ev_population_clean.to_csv("../data/processed/ev_population_clean.csv", index=False)

In [27]:
print("Original rows:", population_df.shape[0])
print("Filtered rows:", ev_population_clean.shape[0])

Original rows: 276828
Filtered rows: 714


### EV Sales Dataset Cleaning:

In [23]:
sales_df.columns

Index(['region_country', 'category', 'parameter', 'mode', 'powertrain', 'year',
       'unit', 'value', 'aggregate group'],
      dtype='object')

In [25]:
sales_df.columns = sales_df.columns.str.lower()

sales_clean = sales_df[[
    "region_country",
    "year",
    "value"
]].rename(columns={"value": "ev_sales"})

In [26]:
sales_clean.isna().sum()   

region_country    0
year              0
ev_sales          0
dtype: int64

In [28]:
print("Original rows:", sales_df.shape[0])
print("Filtered rows:", sales_clean.shape[0])

Original rows: 16436
Filtered rows: 16436


In [29]:
sales_clean.to_csv("../data/processed/ev_sales_clean.csv", index=False)

### Charging Infrastructure Dataset Cleaning:

In [30]:
charging_df.columns

Index(['IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'DataProviderID', 'OperatorID', 'UsageTypeID', 'Connections',
       'NumberOfPoints', 'StatusTypeID', 'DateLastStatusUpdate',
       'DataQualityLevel', 'DateCreated', 'SubmissionStatusTypeID',
       'AddressInfo.ID', 'AddressInfo.Title', 'AddressInfo.AddressLine1',
       'AddressInfo.Town', 'AddressInfo.StateOrProvince',
       'AddressInfo.Postcode', 'AddressInfo.CountryID', 'AddressInfo.Latitude',
       'AddressInfo.Longitude', 'AddressInfo.DistanceUnit',
       'AddressInfo.AddressLine2', 'AddressInfo.RelatedURL', 'UsageCost',
       'GeneralComments', 'AddressInfo.ContactTelephone1',
       'AddressInfo.ContactTelephone2', 'AddressInfo.AccessComments',
       'AddressInfo.ContactEmail', 'DataProvidersReference', 'DatePlanned',
       'country_code', 'OperatorsReference', 'DateLastConfirmed',
       'MetadataValues'],
      dtype='object')

In [31]:
charging_clean = charging_df[[
    "AddressInfo.CountryID",
    "AddressInfo.Latitude",
    "AddressInfo.Longitude",
    "NumberOfPoints"
]].copy()

charging_clean.columns = [
    "country",
    "latitude",
    "longitude",
    "charging_points"
]

In [32]:
charging_clean.isna().sum()

country                0
latitude               0
longitude              0
charging_points    20000
dtype: int64

In [33]:
print("Original rows:", charging_df.shape[0])
print("Filtered rows:", charging_clean.shape[0])

Original rows: 38219
Filtered rows: 38219


In [35]:
charging_clean["charging_points"] = charging_clean["charging_points"].fillna(1)

In [36]:
charging_clean["charging_points"] = charging_clean["charging_points"].astype(int)

In [37]:
charging_clean.isna().sum()

country            0
latitude           0
longitude          0
charging_points    0
dtype: int64

In [38]:
charging_clean.to_csv(
    "../data/processed/charging_infrastructure_clean.csv",
    index=False
)

### Stock Market Dataset Cleaning:

In [39]:
stock_df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Company'],
      dtype='object')

In [40]:
stock_df.isna().sum()

Date            0
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
Company         0
dtype: int64

In [41]:
ev_companies = ["TSLA", "NVDA", "NIO", "XPEV", "LI"]

stock_clean = stock_df[stock_df["Company"].isin(ev_companies)]

In [42]:
stock_clean["Date"] = pd.to_datetime(stock_clean["Date"])

/var/folders/fp/w6yn5n6d4cb299k1w5hy5xgc0000gn/T/ipykernel_91314/3248918533.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  stock_clean["Date"] = pd.to_datetime(stock_clean["Date"])
/var/folders/fp/w6yn5n6d4cb299k1w5hy5xgc0000gn/T/ipykernel_91314/3248918533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock_clean["Date"] = pd.to_datetime(stock_clean["Date"])


In [43]:
stock_clean.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Company'],
      dtype='object')

In [44]:
stock_clean.to_csv("../data/processed/stock_market_clean.csv", index=False)